<a href="https://colab.research.google.com/github/noorahmedbhatti007-byte/US-Airline-Flight-Delays-PySpark/blob/main/BAN_5600_Team_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Uncomment on Google Colab or any machine without PySpark:
!pip install -q pyspark

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import LinearSegmentedColormap

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.functions import vector_to_array

try:
    display  # available in Jupyter / Colab
except NameError:
    display = print

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.environ["FLIGHTS_DATA_DIR"] = "/content/drive/MyDrive/Colab Notebooks/Spark Assignment"

DATA_DIR = os.getenv("FLIGHTS_DATA_DIR", "data")
OUTPUT_DIR = os.getenv("FLIGHTS_OUTPUT_DIR", "outputs")
FIG_DIR, TABLE_DIR = os.path.join(OUTPUT_DIR, "figures"), os.path.join(OUTPUT_DIR, "tables")

DELAY_THRESHOLD = 15
SEED = 42
MODEL_SAMPLE_FRACTION = float(os.getenv("MODEL_SAMPLE_FRACTION", "0.20"))
TUNING_SAMPLE_FRACTION = float(os.getenv("TUNING_SAMPLE_FRACTION", "0.30"))
MIN_FLIGHTS_AIRPORT = int(os.getenv("MIN_FLIGHTS_AIRPORT", "5000"))
MIN_FLIGHTS_ROUTE = int(os.getenv("MIN_FLIGHTS_ROUTE", "1000"))
TE_SMOOTHING = 50
DRIVER_MEMORY = os.getenv("SPARK_DRIVER_MEMORY", "8g")

for d in (FIG_DIR, TABLE_DIR):
    os.makedirs(d, exist_ok=True)